In [1]:
%pip install --quiet -U langgraph typing_extensions langchain_openrouter dotenv exa_py jinja2

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [3]:
from typing import Literal, TypedDict

from pydantic import BaseModel, Field


class Evidence(BaseModel):
    timestamp: str | None = Field(
        default=None,
        description="Timestamp from the transcript, e.g. '00:24'.",
    )
    speaker: Literal[
        "patient",
        "clinician",
        "device_report",
        "unknown",
    ] = Field(
        description="Source of the information."
    )
    excerpt: str = Field(
        description=(
            "Short source excerpt supporting the extracted fact. "
            "Preserve the meaning of the original wording."
        )
    )


class TextFact(BaseModel):
    text: str = Field(
        description="One atomic clinical fact, without additional interpretation."
    )
    evidence: list[Evidence] = Field(default_factory=list)


class SymptomFact(BaseModel):
    symptom: str = Field(
        description="Standard concise name of the symptom."
    )
    anatomical_site: str | None = Field(
        default=None,
        description="Anatomical site when stated.",
    )
    laterality: str | None = Field(
        default=None,
        description=(
            "Left, right, bilateral, or another explicitly stated laterality. "
            "Do not infer laterality."
        ),
    )
    status: Literal["present", "denied", "uncertain"] = Field(
        description=(
            "Whether the symptom is present, explicitly denied, or uncertain."
        )
    )
    course: str | None = Field(
        default=None,
        description=(
            "Change over time, e.g. improved, worsened, intermittent, "
            "migrated, or unchanged."
        ),
    )
    severity: str | None = Field(
        default=None,
        description="Severity exactly as stated, when available.",
    )
    timing: str | None = Field(
        default=None,
        description="Onset, duration, frequency, or time-of-day pattern.",
    )
    evidence: list[Evidence] = Field(default_factory=list)


class MedicationFact(BaseModel):
    name: str | None = Field(
        default=None,
        description=(
            "Actual medication name only when stated. "
            "Use null when the name was not provided."
        ),
    )
    description: str = Field(
        description=(
            "Medication or treatment description, e.g. "
            "'antibiootikum, nimetus mainimata'."
        )
    )
    status: Literal[
        "current",
        "completed",
        "planned",
        "stopped",
        "uncertain",
    ]
    dosage: str | None = None
    frequency: str | None = None
    course_day: int | None = Field(
        default=None,
        description="Current day of the treatment course when explicitly stated.",
    )
    planned_total_days: int | None = Field(
        default=None,
        description="Planned total course duration when explicitly stated.",
    )
    response: str | None = Field(
        default=None,
        description="Reported response to treatment.",
    )
    adverse_effects: list[str] = Field(
        default_factory=list,
        description="Explicitly reported or denied adverse effects.",
    )
    evidence: list[Evidence] = Field(default_factory=list)


class MeasurementFact(BaseModel):
    measurement: str = Field(
        description="Measurement name, e.g. kehatemperatuur or pulss."
    )
    value: str = Field(
        description="Value or range as stated."
    )
    unit: str | None = Field(
        default=None,
        description="Unit when stated.",
    )
    context: str | None = Field(
        default=None,
        description=(
            "Context such as daytime, overnight, smartwatch, at home, "
            "or usual baseline."
        ),
    )
    comparison: str | None = Field(
        default=None,
        description="Comparison with baseline or previous measurement.",
    )
    evidence: list[Evidence] = Field(default_factory=list)


class LaboratoryResultFact(BaseModel):
    analysis: str
    value: str
    unit: str | None = None
    date: str | None = None
    evidence: list[Evidence] = Field(default_factory=list)


class BackgroundFact(BaseModel):
    category: Literal[
        "comorbidity",
        "allergy",
        "past_procedure",
        "past_investigation",
        "family_history",
        "social_history",
        "alcohol_use",
        "tobacco_use",
        "previous_episode",
        "other",
    ]
    text: str
    evidence: list[Evidence] = Field(default_factory=list)


class ExamFinding(BaseModel):
    anatomical_site: str = Field(
        description="Body site examined."
    )
    finding: str = Field(
        description="One atomic examination finding."
    )
    status: Literal["present", "absent", "uncertain"] = Field(
        description="Whether the finding is present, absent, or uncertain."
    )
    laterality: str | None = Field(
        default=None,
        description="Laterality only when explicitly established.",
    )
    comparison: str | None = Field(
        default=None,
        description="Comparison with the opposite side or previous examination.",
    )
    evidence: list[Evidence] = Field(default_factory=list)


class AssessmentFact(BaseModel):
    condition: str = Field(
        description="Condition explicitly discussed by the clinician."
    )
    status: Literal[
        "confirmed",
        "working_diagnosis",
        "differential",
        "considered_less_likely",
        "ruled_out",
        "uncertain",
    ]
    rationale: str | None = Field(
        default=None,
        description=(
            "Explicit clinician reasoning supporting the assessment status."
        ),
    )
    evidence: list[Evidence] = Field(default_factory=list)


class OrderFact(BaseModel):
    category: Literal[
        "laboratory",
        "imaging",
        "procedure",
        "referral",
        "other",
    ]
    name: str = Field(
        description="One ordered or planned test, procedure, or referral."
    )
    status: Literal["planned", "ordered", "completed", "cancelled"]
    timing: str | None = None
    evidence: list[Evidence] = Field(default_factory=list)


class PlanAction(BaseModel):
    category: Literal[
        "medication",
        "self_care",
        "follow_up",
        "safety_netting",
        "monitoring",
        "other",
    ]
    action: str = Field(
        description="One explicit agreed or instructed action."
    )
    timing: str | None = Field(
        default=None,
        description="Timing or duration when stated.",
    )
    condition: str | None = Field(
        default=None,
        description="Condition under which the action should occur.",
    )
    evidence: list[Evidence] = Field(default_factory=list)


class ClinicalExtraction(BaseModel):
    encounter_type: str | None = Field(
        default=None,
        description=(
            "Encounter type when explicit, e.g. korduv vastuvõtt."
        ),
    )

    reasons_for_contact: list[TextFact] = Field(default_factory=list)

    symptoms: list[SymptomFact] = Field(default_factory=list)

    current_and_recent_medications: list[MedicationFact] = Field(
        default_factory=list
    )

    measurements: list[MeasurementFact] = Field(default_factory=list)

    laboratory_results: list[LaboratoryResultFact] = Field(
        default_factory=list
    )

    background_history: list[BackgroundFact] = Field(
        default_factory=list
    )

    physical_exam_findings: list[ExamFinding] = Field(
        default_factory=list
    )

    assessments: list[AssessmentFact] = Field(
        default_factory=list
    )

    orders: list[OrderFact] = Field(default_factory=list)

    plan_actions: list[PlanAction] = Field(default_factory=list)

    unresolved_or_missing_details: list[str] = Field(
        default_factory=list,
        description=(
            "Clinically important ambiguities or missing details, "
            "such as an unnamed medication or unstated laterality."
        ),
    )


class WrittenNote(BaseModel):
    anamnesis: str
    physical_exam: str | None = None
    assessment: str | None = None
    plan: str | None = None




class WrittenNote(BaseModel):
    anamnesis: str = Field(
        description=(
            "Final anamnesis section in Estonian, based only on the structured extraction."
        )
    )

    physical_exam: str | None = Field(
        default=None,
        description=(
            "Final physical examination section in Estonian. "
            "Include only examination findings supported by the extraction."
        ),
    )

    decision: str | None = Field(
        default=None,
        description=(
            "Final clinical decision section in Estonian, including the clinician's "
            "assessment, ordered investigations, treatment, recommendations, "
            "follow-up and safety-netting."
        ),
    )

class NoteState(TypedDict, total=False):
    transcript: str
    encounter_date: str
    extraction: ClinicalExtraction
    written_note: WrittenNote

In [4]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="openai/gpt-5.3-chat",
    temperature=0
)

extraction_model = model.with_structured_output(
    ClinicalExtraction
)

writing_model = model.with_structured_output(
    WrittenNote
)


In [5]:
from langchain_core.messages import HumanMessage, SystemMessage


def extract_clinical_data(state: NoteState) -> dict:
    system_message = SystemMessage(
        content="""
You are a clinical information extraction system.

Your task is to extract clinically relevant facts from a patient-clinician
transcript. You are not writing a clinical note and you must not produce
polished narrative prose.

Rules:

1. Extract only information supported by the transcript.
2. Represent information as atomic facts.
3. Preserve uncertainty, negation, temporality and change over time.
4. Distinguish patient-reported information from clinician-observed findings.
5. Do not turn a clinician's general explanation into a patient-specific fact.
6. Do not infer a diagnosis from symptoms alone.
7. Record a diagnosis only when it is explicitly discussed by the clinician,
   and preserve whether it is confirmed, suspected, differential or less likely.
8. Do not infer left or right laterality when it is not stated.
9. Do not infer an exact calendar date from relative expressions unless the
   encounter date makes the conversion unambiguous.
10. An ordered laboratory test is not a laboratory result.
11. Extract unnamed treatments. When a medication class or generic description
    is mentioned but its name is absent, set the medication name to null and
    preserve the description.
12. Empty lists mean that the information was not mentioned. They do not mean
    that the patient definitively does not have the condition.
13. Every extracted fact should contain a short supporting excerpt and timestamp
    when available.
14. Use Estonian for normalized clinical labels. Preserve source meaning and
    do not improve or embellish the clinical content.
"""
    )

    human_message = HumanMessage(
        content=f"""
Encounter date: {state.get("encounter_date", "not provided")}

Transcript:
{state["transcript"]}
"""
    )

    extraction = extraction_model.invoke(
        [system_message, human_message]
    )

    return {
        "extraction": extraction
    }


def write_clinical_note(state: NoteState) -> dict:
    extraction = state["extraction"]

    system_message = SystemMessage(
        content="""
You are an experienced Estonian physician writing a clinical note.

Write the note using only the supplied structured clinical extraction.

The final note must contain exactly these sections:

1. anamnesis
2. physical_exam
3. decision

Section rules:

ANAMNESIS
- Describe the reason for contact and relevant history.
- Include symptoms, their onset, course, severity and associated findings.
- Include relevant treatment response, home measurements and background history.
- Include patient-reported negative findings when clinically relevant.
- Do not include physical examination findings.
- Do not include the clinician's management plan.

PHYSICAL_EXAM
- Include only findings obtained or explicitly documented during the examination.
- Distinguish observable findings from the patient's response to examination.
- Do not infer normal findings.
- Return null when no physical examination was documented.

DECISION
- Include the clinician's assessment and diagnostic uncertainty.
- Include ordered investigations.
- Include medication decisions, recommendations, follow-up and safety-netting.
- Do not promote a differential diagnosis into a confirmed diagnosis.
- Return null when no decision or plan was documented.

General rules:

- Do not introduce information absent from the extraction.
- Preserve uncertainty, negation and temporality.
- Do not invent diagnoses, medication names, dates or laterality.
- Do not treat missing information as a negative finding.
- Merge repeated facts without losing clinically relevant detail.
- Do not include timestamps, evidence excerpts or speaker labels.
- Write concise, natural and clinically appropriate Estonian.
"""
    )

    human_message = HumanMessage(
        content=f"""
Structured clinical extraction:

{extraction.model_dump_json(
    indent=2,
    exclude_none=True,
)}
"""
    )

    written_note = writing_model.invoke(
        [system_message, human_message]
    )

    return {
        "written_note": written_note
    }

In [6]:
from langgraph.graph import StateGraph, START, END


builder = StateGraph(NoteState)

builder.add_node(
    "extract_clinical_data",
    extract_clinical_data,
)

builder.add_node(
    "write_clinical_note",
    write_clinical_note,
)

builder.add_edge(
    START,
    "extract_clinical_data",
)

builder.add_edge(
    "extract_clinical_data",
    "write_clinical_note",
)

builder.add_edge(
    "write_clinical_note",
    END,
)

graph = builder.compile()

In [12]:
transcript = "[00:00] Nii, aga siis selle seadeldisega lindistame. See jääb teile. Ma korra ütlen lihtsalt, et meil on korduv vastuvõtt, onju. Ja täna on antibiootikumi üheksas päev, mina vaatasin. Neljapäev alustasime jah, kahekümne esimesel siis on kaheksas päev, onju. [00:18] Täna hommikul sai kaheksa nagu täis, siis õhtul läheks üheksas. Kuidas nüüd seis on? [00:24] See valu on suuremalt osalt kadunud, aga õrnalt jääb ikka. Pigem selline pakitsus ja nagu pingetunne. Ja see punetus, noh, vahel nagu lööb tagasi, eriti õhtul. [00:38] Turse on kuidas, kas on tagasi tõmmanud või pigem sama? [00:42] Ütleme, jala pealt on natuke vähem, varbaid saab paremini liigutada. Aga hüppeliigese juures on nagu rohkem tulnud ja tundub, et läheb veidi ülespoole säärde. [00:54] Et teile tundub, et ta on nagu liikunud üles, aga samas üldine jämedus on veidi vähem? [01:00] Jah, täpselt. Ei ole enam nii “kõva” turse, aga ikka paras jäme on. [01:07] Okei. Kas vahepeal on palavikku ka olnud või külmavärinaid? [01:12] Öösel nagu oli, ma ei tea, kas päris palavik, aga temperatuur on tõusnud. Mul on see nutiseade… kell, mis salvestab öiseid näite. [01:24] Te mõõtsite siis kellaga, jah? [01:26] Jah. Oih vabandust, siin ta näitab. Eelmine nädal oli madalam ja siis paar ööd järjest tõusis, mingi null koma seitse, siis üks koma üks, siis üks koma kaks või nii. [01:40] Kas ta näitab ka, mis see baseline on, millega ta võrdleb? [01:44] Baseline’i ma ei ole sisse pannud, ta ise arvutab. Mul endal baseline on tavaliselt alla 36, mingi 35,8 umbes. [01:53] No see ongi huvitav, et öösiti kehatemperatuur tavaliselt langeb, ja kui ta pigem tõuseb, siis midagi kehas toimub. Aga päris palavikuks me üle 37,5 pigem loeme, eks. Päeval mõõtsite ka? [02:07] Päeval kraadisin, oli 36,9 ja üks kord 37,0. Nii et mitte kõrge, aga nagu kõrgem kui muidu. [02:16] Okei. Muud näitajad, pulss või hingamine, kas kell näitab ka midagi “väljas ringest”? [02:22] Jah, ta näitab, et taastumine või midagi on kehvem. Pulss on natuke kõrgem, mul tavaliselt on 54, nüüd oli öösel 60 ringis. [02:32] Infektsiooni või põletikuga võib pulss tõusta küll, kuigi numbriliselt on see ikkagi madal. Muud liigesed ei valuta praegu? [02:40] Ei, muud liigesed ei valuta. [02:43] Antibiootikumiga kõrvaltoimeid, kõhuga muret, löövet? [02:47] Ei, absoluutselt ei ole. Kõht on okei. [02:52] Eelmine kord teil läks ka päris pikalt enne kui jalg rahunes, eks? [02:56] Jah, tookord oli vist peaaegu kuu aega. Aga siis ma ei pidanud teist antibiootikumi juurde võtma. [03:03] Siis tegime selle pika kümnepäevase kuuri ja jäi vaikselt maha. Praegu ma vaataks selle jala üle ja laseks veenivereanalüüsi võtta: üldvere, põletikunäitajad, ja igaks juhuks kusihape ka. Kuigi kui on podagra atakk, siis kusihape võib olla hoopis madalam ja see analüüs pole 100% usaldusväärne. [03:24] Ma ise ka ei arva, et podagra, sest see algas pigem siit talla juurest ja siis tuli üles. Aga noh, parem kontrollida. [03:32] Just. Olge hea, võtke sokid ja jalanõud ära, mõlemalt jalalt, siis on hea võrrelda. Ja heitke pikali, mul on lihtsam vaadata. [03:43] Jah. Ma natuke kartsin, et kui ma ei saa millegagi hakkama… et talvel on keeruline, kui jalg nii turses. [03:51] No jah, talvel on see eriti tüütu. Okei, ma vaatan… värv on täna parem kui oli? [03:59] On küll, eelmine nädal oli tumepunane, peaaegu tulipunane. Mul isegi pilt on telefonis. [04:06] Turse ulatub tegelikult siia säärde ka, onju. Ja katsudes on soojem kui teine jalg, jah. Siit vajutades… kas siit on valus? [04:17] Siit ei ole. Siit külje pealt ka eriti ei ole. [04:22] Liigutades hüppeliigest, kas teeb haiget? [04:25] Ei, liigutades ei ole. Ainult pöidla all, nagu päka juures, on natuke hell. [04:32] Jah, siin on kõige rohkem turses ka. Siit alt… okei. Mujalt väga ei valuta. Hea. Võite tagasi panna. [04:43] Mulle endale tundub, et ta on isegi natuke rohkem turses kui eelmisel korral, kui ma käisin. [04:49] Eelmine kord, jah, kui te käisite [spetsialist] juures ja siis minu juures ka. Siis see alles algas, te jõudsite kiiresti arstile, see on hea. [04:59] No seda rohtu ma praegu ei tahaks kohe vahetada, kuigi võib-olla on seda vaja. Teeme enne analüüsid ära ja otsustame selle alusel. Praegu võtate edasi, kokku 12 päeva, nii et umbes kolm päeva on veel minna, lõpeb vist laupäeva õhtul või pühapäeva hommikul. [05:16] Okei. [05:18] Nii, analüüsidest: hemogramm, CRP ja muud põletikunäitajad, ja kusihape. Kui me juba torkame, siis teeme ära. Ma vaatan kohe, kuhu te saate minna. [05:30] Üks hetk… lähete kabinet neli. Ma annan õele märku ka, et te tulete. Veeniveri siis. [05:39] Hästi. [05:40] Ja homme ma helistan vastustega, kas sobib? Homseks peaks olemas olema. [05:45] Sobib küll. [05:47] Okei. Siis kabinet neli, analüüsid. Praegu võtate antibiootikumi edasi. Jalga hoiate võimalusel üleval, kui istute, siis pigem sirgelt ja natuke kõrgemal. Ja jahedat võib peale panna, mitte jääkülma, aga siukest jahedat kompressi. [06:05] Ma olen õhtuti teinud soolvee leotist, sellist sooja… see vist ei ole hea? [06:10] Soe võib turset süvendada, jah. Kui te tunnete, et leevendab, siis võib, aga pigem soovitatakse jahedat. Ja see on ka oluline, et kui jalg on üleval ja siis lasete alla, siis veri nagu vajub jalga tagasi ja siis võib olla kehvem hetk. [06:28] Arusaadav. Ma siis proovin rohkem jahedat ja hoian üleval. [06:33] Just. Olge nii, nii mugavalt kui võimalik, piinlema ei pea. Kas midagi veel küsida praegu? [06:40] Ei, praegu rohkem ei ole. Siis räägime homme. [06:44] Hästi. Selle ma panen kinni."
transcript_1 = "00:00] Arst: Kuulan, millest me räägime täna. [00:04] Patsient: Me räägime sellest, et mul on nüüd jälle selline ebamugav tunne siin paremal alakõhus, aga natuke teistmoodi kui kevadel. Nagu pigem selline torkiv-suruv, ja vahel tekib pärast söömist või kui olen pikemalt istunud. Kevade poole käisin, siis nagu läks paremaks, tegin füsioterapeudi juures harjutusi ja vähendasin trennis koormust. Mulle tundus et läks ära, aga nüüd on ikkagi tagasi. [00:33] Arst: Kas te saate aru, mis ajast see uuesti ägenes või millal te viimati mõtlesite, et on päris korras? [00:40] Patsient: Ma ei oskagi täpselt. Võib-olla suve lõpus oli nagu rahulikum, ja siis mingi viimase kuu jooksul on jälle rohkem meelde tuletanud. Ta ei ole nagu väga valus, pigem ebameeldiv. Kui skaalal, siis mingi 1–2, max 3. [01:00] Arst: Kas ta on kogu aeg olemas või pigem käib ja läheb? [01:04] Patsient: Pigem käib ja läheb. Näiteks söön, ja siis natuke aega hiljem tunnen seal nagu midagi… ja siis võib terve päev mitte olla. Täna vist polegi eriti olnud. [01:18] Arst: Kui ta tekib, kui kaua ta kestab? Minutid või võib olla ka tunde? [01:24] Patsient: Minutid pigem. Tunde ma ei usu. Ta on selline väike, ja harjud ära ka, kui ta ei ole hirmus häiriv. [01:36] Arst: Kas mingi asend või liigutus provotseerib? Istumine, püsti, lamamine, töö, trenn? [01:43] Patsient: Ma ei seostaks väga. Trennis pigem ei ägene, ma saan rahulikult teha. Ainult kui ma nagu väga kõvasti pingutan kõhtu või jalgu, siis võib korraks tunda. Aga pärast trenni hullemaks ei lähe. [02:02] Arst: Kas olete midagi võtnud, valuvaigisteid või midagi? [02:06] Patsient: Ei, pole vaja olnud. [02:09] Arst: Öösel segab? Ärkate selle peale? [02:12] Patsient: Ei, ma magan niigi kehvasti, ma saaks aru kui see segaks, aga ei ole. [02:18] Arst: Kuidas kõht läbi käib? Kõhulahtisust, kõhukinnisust, verd väljaheites? [02:24] Patsient: Täitsa tavaliselt. Ei ole lahti ega kinni. Verd pole märganud. [02:30] Arst: Iiveldust, oksendamist? [02:32] Patsient: Ei. [02:34] Arst: Kõrvetisi, maohappe ülesviskamist? [02:37] Patsient: Ei, seda ka mitte. [02:40] Arst: Kuidas urineerimisega on? Kas on kipitust, sagenemist, öösel käimist? [02:47] Patsient: Üldiselt okei. Aga mõnikord pärast pissimist on nagu korraks selline “täksti” tunne seal samas piirkonnas. Ja kui ma pean kaua kinni hoidma, siis nagu ka annab tunda. [03:03] Arst: Uriinis verd pole näinud? [03:06] Patsient: Ei. [03:08] Arst: Erektsioon, ejakulatsioon, kas sellega seostub midagi? [03:12] Patsient: Ei oska öelda, pigem ei. Ainult see urineerimise teema vahel. [03:18] Arst: Munandites valu ei ole? [03:20] Patsient: Ei ole, kõik korras. [03:23] Arst: Kas valu kiirgub kuskile, selga või kubemesse? [03:27] Patsient: Pigem ei. Selg vahel harva annab tunda, aga ma arvan et see on eraldi. [03:35] Arst: Ma vaatan korra eelmise korra märkmeid… Me rääkisime juba jaanuaris, jah, varsti saab aasta. Siis oli juttu ka, et autoga sõites tekib. Kas praegu on sama? [03:49] Patsient: Ausalt ma ei tea, ma pole tähele pannud. Võib-olla istumisega natuke, aga ei ole kindel. [03:56] Arst: Ultraheli oli tookord tehtud, eks? [03:59] Patsient: Jah. [04:01] Arst: Siin on vastus, oli korras. Analüüsidest oli üldveri, põletikunäitaja ja uriin ka vaadatud. Okei. Täna ma teeks paar lisaanalüüsi, kui te olete nõus. Kolesterooli ja veresuhkrut ei pea, maksanäitajad olid sügisel tehtud, aga uriinianalüüsi tahaks uuesti ja paar vereproovi ka. [04:25] Patsient: Jah, sobib. [04:27] Arst: Neerukive varem teada ei ole? Mingeid urotrakti kive, niiöelda? [04:32] Patsient: Ei, ma ei tea. [04:34] Arst: Kõhu piirkonnas operatsioone pole olnud? [04:37] Patsient: Ei ole. [04:39] Arst: Nii. Ma kohe ei oska öelda, mis see täpselt olla võiks, sest see on taustal, ebamäärane, ja huvitav on see, et pärast sööki vahel ägeneb, aga samas koht on alakõhus. Eelmine kord rääkisime ka songast, aga see ei jäänud kahtlaseks. Ultraheli ka ei näidanud. [05:02] Patsient: Jah, täpselt. Ma ise ka ei oska seda kuidagi paremini seletada, ma nagu proovin oma peas kokku panna, aga… ei tule. [05:11] Arst: Okei. Kas teil on veel midagi, mida tahate näidata või mainida? [05:15] Patsient: Jah, üks asi on veel. Mul on siin kõhu peal nagu üks koht, mis vahel sügeleb ja on nagu punn, ja mulle tundub et ta on suuremaks läinud. Ma ei tea kas see on mingi lööve või… [05:30] Arst: Pigem üksik moodustis? Millal te esimest korda märkasite? [05:35] Patsient: Ma ei tea. Võib-olla mingi väike asi oli ammu, aga nüüd viimase paari kuu jooksul on nagu rohkem silma jäänud. Ja vahel sügeleb. Midagi määrinud ei ole. [05:48] Arst: Allergiaid teada? Vigastanud kogemata ei ole? [05:52] Patsient: Ei ole allergiaid, ja vigastanud ka vist mitte. [05:56] Arst: Okei. Vaatame üle. Kõigepealt vaatan seda nahamuutust, siis palun teid pikali ja katsun kõhu läbi. [06:05] Patsient: Okei. [06:07] Arst: See näeb välja nagu seborröa keratoos, see on healoomuline nahamuutus. Sellega ei pea midagi tegema. Kui sügeleb, siis apteegist käsimüügist hüdrokortisoonkreem, õhuke kiht 2x päevas mõni päev. [06:25] Patsient: Ta praegu näiteks ei sügele, aga jah, vahel on. Ikka natuke hirmus, kui igasugused asjad tekivad. [06:33] Arst: Mõistan. Nii, võtke palun jalanõud ära, vaatame kaalu üle. [06:38] Patsient: Võib-olla olen kilo juurde võtnud, ma vahepeal trennis kaalusin. [06:43] Arst: Aasta tagasi oli 76,8, praegu on natukene rohkem jah. Nüüd palun pluus ära, püksirihm lahti ja heitke siia pikali. [06:55] Patsient: Jah. [06:58] Arst: Pange jalad põlvedest kõveraks. Ma katsun kõhtu. Kui kuskil on valu või imelik tunne, ütlete kohe. [07:07] Patsient: Niimoodi pindmiselt ei ole. [07:10] Arst: Lähen sügavamalt… [07:13] Patsient: Ei ole midagi hullu. [07:15] Arst: Nüüd pange jalad sirgeks. Siit paremal vaagna piirkonnas… kas siin on teistmoodi? [07:23] Patsient: Jah, seal on see koht. [07:26] Arst: Kui ma vajutan ja lasen lahti, kumb on valusam, vajutamine või lahtilaskmine? [07:32] Patsient: Vajutamine. Lahtilaskmine ei ole valus. [07:36] Arst: Okei. Võite püsti tulla. Ma annan teile ühe lühikese testi ka täita, mis küsib urineerimise kohta. [07:46] Patsient: Ma ei oska mõnele küsimusele vastata, näiteks et kas põis saab täielikult tühjaks… mis tunne see peaks üldse olema. [07:55] Arst: No tavaliselt kui ei saa, siis inimene käib ära, tõuseb püsti ja tekib tunne, et “oi, peaks veel”. Kui seda pole, siis pigem tühjeneb. Öine käimine sõltub ka, palju joote ja kas kohvi või alkoholi on. [08:10] Patsient: Jah, kohv ja alko, siis on jah eriti. Ma vist märkisin siia “peaaegu alati”, aga tegelikult mõtlesin “peaaegu mitte kunagi”, vabandust. [08:21] Arst: Selge, ma panen õigesti kirja. Kokku tuleb siin 2 punkti, väga suuri kaebusi ei tule. [08:30] Patsient: Mis me siis edasi teeme? [08:33] Arst: Teeme täna uriinianalüüsi, kui teil õnnestub kohe anda. Vereanalüüsis panen neeru- ja põletikunäitajad, ja PSA ka igaks juhuks, see on eesnäärme analüüs. Vaatame, et midagi varjatut ei oleks. [08:52] Patsient: Okei. [08:54] Arst: Kui kõik on korras, siis see on selline väike vaagnavalu-alakõhu valu, millega tegelevad ka meestearstid. Teine variant on sisearst. Ma ootaks analüüsid ära ja siis otsustan suuna. Vajadusel teen e-konsultatsiooni eriarstile, et mida veel uurida. Perearstina ma näiteks kompuutrit ise tellida ei saa, aga eriarst saab vajadusel. [09:19] Patsient: Selge, ma proovin ka ise jälgida, et millal tekib ja mis enne toimus. [09:25] Arst: Just, kasvõi mõttes päevikut pidada: söök, istumine, trenn, vedelik, urineerimine. Äkki tuleb mingi seos välja. Ja oluline on ka see, et võib-olla ei olegi midagi tõsist, aga uurime ära, et välistada. [09:44] Patsient: Jah, ma olen seda oma peas teinud, aga proovin teadlikumalt. [09:49] Arst: Nii, käige palun tualetis läbi ja tooge natuke uriini, palju pole vaja. Siis minge kabinetti number neli, sealt võetakse veri ka. [10:01] Patsient: Registratuurist öeldi ka, et peaks neljast läbi käima, nii et sobib. [10:06] Arst: Homme on analüüsid valmis. Sobib, kui ma helistan homme ja räägime tulemused ning järgmise sammu? [10:13] Patsient: Jah, sobib. [10:15] Arst: Ja infoks veel: gripivaktsiin on võimalik teha, 18 eurot, kaitse üheks hooajaks. Mina soovitan. [10:23] Patsient: Ma võin teha, jah. Kui juba olen siin, siis miks mitte. [10:28] Arst: Väga hea. Öelge õele ka, ta saab selle ära teha koos analüüsidega. Ma annan talle märku. [10:35] Patsient: Okei, aitäh."
result = graph.invoke(
    {
        "transcript": transcript_1,
        "encounter_date": "27.11.2025",
    }
)

In [13]:
from IPython.display import Markdown, display


def display_written_note(note: WrittenNote) -> None:
    markdown = f"""
## Anamnees

{note.anamnesis}

## Füüsiline läbivaatus

{note.physical_exam or "_Ei ole dokumenteeritud._"}

## Otsus

{note.decision or "_Ei ole dokumenteeritud._"}
"""

    display(Markdown(markdown))


display_written_note(result["written_note"])


## Anamnees

Patsient pöördub korduva parema alakõhu ebamugavustunde tõttu ning nahal esineva sügeleva punniga kõhul. Parema alakõhu valu on esinenud varem ning viimase kuu jooksul sagedamini. Valu on vahelduv, kestab pigem minuteid, tugevus 1–2/10, maksimaalselt kuni 3/10. Tekib vahel pärast söömist, võimalik seos pikema istumisega ei ole kindel. Tugeval kõhu või jalgade pingutusel võib valu korraks esile tulla. Valu ei kiirgu. Öist valu ei esine. Kaasuvatest sümptomitest eitab kõhulahtisust, kõhukinnisust, verirooja, iiveldust, oksendamist ja kõrvetisi. Esineb aeg-ajalt lühiajaline ebamugavustunne samas piirkonnas pärast urineerimist või uriini kinni hoidmisel. Hematuuriat ei esine. Munandivalu ei ole.

Nahal kõhul üksik punn, mis sügeleb vahelduvalt, märgatavam viimase paari kuu jooksul.

Valuvaigisteid ei ole tarvitanud. Varasemalt tehtud kõhu ultraheli ning vere- ja uriinianalüüsid on olnud korras. Neerukive teada ei ole. Kõhupiirkonna operatsioone pole olnud. Allergiaid ei esine. Kehakaal on võrreldes eelmise aastaga veidi tõusnud.

## Füüsiline läbivaatus

Kõhu pindmisel palpatsioonil valu ei esine. Paremas vaagnapiirkonnas palpatsioonil hellus, valu tugevam vajutamisel kui lahtilaskmisel. Kõhu nahal üksik seborröa keratoosile vastav moodustis.

## Otsus

Kliiniline pilt sobib ebaselge põhjusega kerge vaagnavalu/alakõhu valuga; tõsine patoloogia hetkel ebatõenäoline, varasemad uuringud korras. Song vähem tõenäoline varasema hindamise ja ultraheli alusel. Nahaline leid vastab seborröa keratoosile (healoomuline).

Tellitud uriinianalüüs ning vereanalüüs (neeru- ja põletikunäitajad, PSA).

Soovitatud pidada sümptomite seoste jälgimiseks päevikut (toitumine, istumine, füüsiline koormus, vedeliku tarbimine, urineerimine).

Sügeluse korral kasutada hüdrokortisoonkreemi õhukese kihina 2x päevas mõne päeva jooksul.

Kokkulepe: arst annab analüüside vastused järgmisel päeval telefoni teel.

Lisaks planeeritud gripivaktsineerimine.


In [9]:
from IPython.display import Markdown, display


def show_clinical_note(note_data: dict) -> None:
    anamnesis = note_data["anamnesis"]
    physical_exam = note_data["physical_exam"]
    decision = note_data["decision"]

    def render_items(items, formatter):
        if not items:
            return "_Ei ole dokumenteeritud._"
        return "\n".join(f"- {formatter(item)}" for item in items)

    comorbidities = render_items(
        anamnesis.get("comorbidities", []),
        lambda x: x["name"],
    )

    medications = render_items(
        anamnesis.get("medications", []),
        lambda x: " ".join(
            part
            for part in [
                x.get("name"),
                x.get("dosage"),
                x.get("frequency"),
            ]
            if part
        ),
    )

    laboratory_results = render_items(
        anamnesis.get("laboratory_results", []),
        lambda x: " ".join(
            part
            for part in [
                x.get("analysis"),
                x.get("value"),
                x.get("unit"),
                x.get("date"),
            ]
            if part
        ),
    )

    home_monitoring = render_items(
        anamnesis.get("home_monitoring", []),
        lambda x: (
            f"**{x['measurement'].capitalize()}:** {x['value']}"
            + (f" — {x['context']}" if x.get("context") else "")
        ),
    )

    family_history = render_items(
        anamnesis.get("family_history", []),
        lambda x: (
            f"{x['relative']}: {x['condition']}"
            + (f" ({x['detail']})" if x.get("detail") else "")
        ),
    )

    markdown = f"""
# Kliiniline märge

## Anamnees

### Pöördumise põhjus

{anamnesis.get("chief_complaint", "_Ei ole dokumenteeritud._")}

### Käesoleva haiguse anamnees

{anamnesis.get("history_of_present_illness", "_Ei ole dokumenteeritud._")}

### Kaasuvad haigused

{comorbidities}

### Ravimid

{medications}

### Laboratoorsed analüüsid

{laboratory_results}

### Kodused mõõtmised

{home_monitoring}

### Perekonnaanamnees

{family_history}

## Füüsiline läbivaatus

{physical_exam.get("findings", "_Ei ole dokumenteeritud._")}

## Hinnang

{decision.get("assessment", "_Ei ole dokumenteeritud._")}

## Plaan

{decision.get("plan", "_Ei ole dokumenteeritud._")}
"""

    display(Markdown(markdown))

In [10]:
import pandas as pd

from IPython.display import HTML, Markdown, display
from pydantic import BaseModel


def visualise_extraction(extraction) -> None:
    """
    Display a ClinicalExtraction Pydantic model or dictionary
    as readable Jupyter tables.
    """

    if isinstance(extraction, BaseModel):
        data = extraction.model_dump(exclude_none=True)
    else:
        data = extraction

    def evidence_text(evidence: list[dict] | None) -> str:
        if not evidence:
            return ""

        parts = []

        for item in evidence:
            timestamp = item.get("timestamp", "")
            speaker = item.get("speaker", "")
            excerpt = item.get("excerpt", "")

            prefix_parts = [
                part for part in [timestamp, speaker]
                if part
            ]

            prefix = " · ".join(prefix_parts)

            if prefix:
                parts.append(f"{prefix}: “{excerpt}”")
            else:
                parts.append(f"“{excerpt}”")

        return "\n".join(parts)

    def clean_rows(items: list[dict]) -> list[dict]:
        rows = []

        for item in items:
            row = {
                key: value
                for key, value in item.items()
                if key != "evidence"
            }

            row["evidence"] = evidence_text(
                item.get("evidence", [])
            )

            rows.append(row)

        return rows

    def display_table(
        title: str,
        items: list[dict],
        rename: dict | None = None,
    ) -> None:
        if not items:
            return

        display(Markdown(f"### {title}"))

        frame = pd.DataFrame(clean_rows(items))

        if rename:
            frame = frame.rename(columns=rename)

        display(
            frame.style.set_properties(
                **{
                    "text-align": "left",
                    "white-space": "pre-wrap",
                    "vertical-align": "top",
                }
            ).hide(axis="index")
        )

    encounter_type = data.get(
        "encounter_type",
        "Ei ole määratud",
    )

    summary_counts = {
        "Sümptomid": len(data.get("symptoms", [])),
        "Ravimid": len(
            data.get("current_and_recent_medications", [])
        ),
        "Mõõtmised": len(data.get("measurements", [])),
        "Läbivaatuse leiud": len(
            data.get("physical_exam_findings", [])
        ),
        "Hinnangud": len(data.get("assessments", [])),
        "Korraldused": len(data.get("orders", [])),
        "Plaani tegevused": len(
            data.get("plan_actions", [])
        ),
    }

    cards = "".join(
        f"""
        <div style="
            border: 1px solid #ddd;
            border-radius: 8px;
            padding: 10px 14px;
            min-width: 120px;
        ">
            <div style="font-size: 12px; color: #666;">
                {label}
            </div>
            <div style="font-size: 22px; font-weight: 600;">
                {count}
            </div>
        </div>
        """
        for label, count in summary_counts.items()
    )

    display(
        HTML(
            f"""
            <div style="margin-bottom: 18px;">
                <h2 style="margin-bottom: 4px;">
                    Kliinilise info ekstraktsioon
                </h2>
                <div style="color: #666; margin-bottom: 14px;">
                    Vastuvõtu tüüp: <b>{encounter_type}</b>
                </div>

                <div style="
                    display: flex;
                    flex-wrap: wrap;
                    gap: 8px;
                ">
                    {cards}
                </div>
            </div>
            """
        )
    )

    display_table(
        "Pöördumise põhjused",
        data.get("reasons_for_contact", []),
        {
            "text": "Põhjus",
            "evidence": "Allikas",
        },
    )

    display_table(
        "Sümptomid",
        data.get("symptoms", []),
        {
            "symptom": "Sümptom",
            "anatomical_site": "Piirkond",
            "status": "Staatus",
            "course": "Kulg",
            "severity": "Raskus",
            "timing": "Ajastus",
            "evidence": "Allikas",
        },
    )

    display_table(
        "Ravimid ja ravi",
        data.get("current_and_recent_medications", []),
        {
            "name": "Ravimi nimi",
            "description": "Kirjeldus",
            "status": "Staatus",
            "dosage": "Annus",
            "frequency": "Sagedus",
            "course_day": "Ravipäev",
            "planned_total_days": "Kuuri pikkus",
            "response": "Ravivastus",
            "adverse_effects": "Kõrvaltoimed",
            "evidence": "Allikas",
        },
    )

    display_table(
        "Mõõtmised",
        data.get("measurements", []),
        {
            "measurement": "Mõõtmine",
            "value": "Väärtus",
            "unit": "Ühik",
            "context": "Kontekst",
            "comparison": "Võrdlus",
            "evidence": "Allikas",
        },
    )

    display_table(
        "Taustaanamnees",
        data.get("background_history", []),
        {
            "category": "Kategooria",
            "text": "Fakt",
            "evidence": "Allikas",
        },
    )

    display_table(
        "Füüsilise läbivaatuse leiud",
        data.get("physical_exam_findings", []),
        {
            "anatomical_site": "Piirkond",
            "finding": "Leid",
            "status": "Staatus",
            "finding_type": "Leiu tüüp",
            "laterality": "Lateraalsus",
            "comparison": "Võrdlus",
            "evidence": "Allikas",
        },
    )

    display_table(
        "Kliinilised hinnangud",
        data.get("assessments", []),
        {
            "condition": "Seisund",
            "status": "Staatus",
            "rationale": "Põhjendus",
            "evidence": "Allikas",
        },
    )

    display_table(
        "Tellitud uuringud",
        data.get("orders", []),
        {
            "category": "Kategooria",
            "name": "Uuring",
            "status": "Staatus",
            "timing": "Ajastus",
            "evidence": "Allikas",
        },
    )

    display_table(
        "Plaan",
        data.get("plan_actions", []),
        {
            "category": "Kategooria",
            "action": "Tegevus",
            "timing": "Ajastus",
            "condition": "Tingimus",
            "evidence": "Allikas",
        },
    )

    unresolved = data.get(
        "unresolved_or_missing_details",
        [],
    )

    if unresolved:
        items = "".join(
            f"<li>{item}</li>"
            for item in unresolved
        )

        display(
            HTML(
                f"""
                <div style="
                    margin-top: 18px;
                    border-left: 4px solid #d97706;
                    background: rgba(217, 119, 6, 0.08);
                    padding: 12px 16px;
                    border-radius: 4px;
                ">
                    <b>Puuduv või ebaselge info</b>
                    <ul style="margin-bottom: 0;">
                        {items}
                    </ul>
                </div>
                """
            )
        )

In [14]:
visualise_extraction(
    result["extraction"]
)

### Pöördumise põhjused

Põhjus,Allikas
parema alakõhu ebamugavustunne kordub,00:04 · patient: “mul on nüüd jälle selline ebamugav tunne siin paremal alakõhus”
nahal üksik sügelev punn kõhul,"05:15 · patient: “Mul on siin kõhu peal nagu üks koht, mis vahel sügeleb ja on nagu punn”"


### Sümptomid

Sümptom,Piirkond,laterality,Staatus,Kulg,Raskus,Ajastus,Allikas
alakõhu valu,parem alakõht,parem,present,"intermittent, recurrent","1–2, max 3",viimase kuu jooksul sagedamini; kestab minutid; vahel pärast söömist või istumist,"00:04 · patient: “ebamugav tunne siin paremal alakõhus” 00:40 · patient: “viimase kuu jooksul on jälle rohkem meelde tuletanud” 01:24 · patient: “Minutid pigem” 00:40 · patient: “Kui skaalal, siis mingi 1–2, max 3” 01:04 · patient: “Pigem käib ja läheb”"
valu pärast söömist,parem alakõht,parem,present,intermittent,nan,pärast söömist,00:04 · patient: “vahel tekib pärast söömist”
valu istumisel,parem alakõht,parem,uncertain,nan,nan,pärast pikemat istumist,"00:04 · patient: “või kui olen pikemalt istunud” 03:49 · patient: “Võib-olla istumisega natuke, aga ei ole kindel”"
valu füüsilisel pingutusel,parem alakõht,parem,present,brief,nan,kõhu või jalgade tugeval pingutusel,"01:43 · patient: “kui ma nagu väga kõvasti pingutan kõhtu või jalgu, siis võib korraks tunda”"
öövalu,nan,nan,denied,nan,nan,öösel,"02:12 · patient: “Ei, ma magan niigi kehvasti... aga ei ole”"
kõhulahtisus,nan,nan,denied,nan,nan,nan,02:24 · patient: “Ei ole lahti”
kõhukinnisus,nan,nan,denied,nan,nan,nan,02:24 · patient: “ega kinni”
veri väljaheites,nan,nan,denied,nan,nan,nan,02:24 · patient: “Verd pole märganud”
iiveldus,nan,nan,denied,nan,nan,nan,02:32 · patient: “Ei”
oksendamine,nan,nan,denied,nan,nan,nan,02:32 · patient: “Ei”


### Ravimid ja ravi

...

Seisund,Staatus,Põhjendus,Allikas
seborröa keratoos,confirmed,kliiniline vaatlus,"06:07 · clinician: “See näeb välja nagu seborröa keratoos, see on healoomuline nahamuutus”"
song,considered_less_likely,varasem hindamine ja ultraheli ei toetanud,"05:02 · clinician: “Eelmine kord rääkisime ka songast, aga see ei jäänud kahtlaseks. Ultraheli ka ei näidanud”"
alakõhu/vaagnavalu ebaselge põhjusega,working_diagnosis,"ebamäärane, kerge, uuringud seni korras",08:54 · clinician: “see on selline väike vaagnavalu-alakõhu valu”


### Tellitud uuringud

Kategooria,Uuring,Staatus,Ajastus,Allikas
laboratory,uriinianalüüs,ordered,täna,08:33 · clinician: “Teeme täna uriinianalüüsi”
laboratory,"vereanalüüs (neeru- ja põletikunäitajad, PSA)",ordered,täna,"08:33 · clinician: “Vereanalüüsis panen neeru- ja põletikunäitajad, ja PSA ka”"


### Plaan

Kategooria,Tegevus,Allikas,Ajastus,Tingimus
monitoring,"pidada sümptomite päevikut (seos söök, istumine, trenn, vedelik, urineerimine)","09:25 · clinician: “kasvõi mõttes päevikut pidada: söök, istumine, trenn, vedelik, urineerimine”",nan,nan
follow_up,arst helistab analüüside vastustega,"10:06 · clinician: “Homme on analüüsid valmis. Sobib, kui ma helistan homme”",homme,nan
medication,kasutada hüdrokortisoonkreemi sügeluse korral,"06:07 · clinician: “Kui sügeleb, siis... hüdrokortisoonkreem”",2x päevas mõni päev,kui sügeleb
other,gripivaktsiin teha,"10:23 · clinician: “gripivaktsiin on võimalik teha” 10:28 · patient: “Ma võin teha, jah”",täna,nan
